In [1]:
# 1. Bibliotēku uzstādīšana
!pip install llama-index llama-index-llms-openai-like llama-index-embeddings-openai llama-index-readers-file pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 6.8 MB/s eta 0:00:00


In [2]:
# 2. API savienojums
from google.colab import userdata
from llama_index.core import Settings
from llama_index.llms.openai_like import OpenAILike
from llama_index.embeddings.openai import OpenAIEmbedding

Settings.llm = OpenAILike(
    model="openai/gpt-3.5-turbo",
    api_key=userdata.get("OPENROUTER_API_KEY"),
    api_base="https://openrouter.ai/api/v1",
    is_chat_model=True
)

Settings.embed_model = OpenAIEmbedding(
    model="text-embedding-ada-002",
    api_key=userdata.get("OPENROUTER_API_KEY"),
    api_base="https://openrouter.ai/api/v1"
)

print("Savienojums veiksmīgs!")

Savienojums veiksmīgs!


In [3]:
# 3. Dokumentu ielāde un indeksēšana
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

dokumenti = SimpleDirectoryReader(
    input_files=[
        "/content/2023_canadian_budget.pdf",
        "/content/ag-studio.pdf",
        "/content/paul_graham_essay.txt"
    ]
).load_data()

indekss = VectorStoreIndex.from_documents(dokumenti)

print(f"Ielādēti {len(dokumenti)} dokumenti")
print("Indekss izveidots!")

Ielādēti 13 dokumenti
Indekss izveidots!


In [ ]:
# Prompts dažādiem jautājumu tipiem

In [11]:
# 4. smart_query() funkcija
from llama_index.core import PromptTemplate

# Prompts dažādiem jautājumu tipiem
PROMPTS = {
    "fact": PromptTemplate(
        "Answer the factual question precisely and concisely.\n"
        "Context: {context_str}\n"
        "Question: {query_str}\n"
        "Answer:"
    ),
    "comparison": PromptTemplate(
        "Compare information from different documents.\n"
        "Context: {context_str}\n"
        "Question: {query_str}\n"
        "Comparison:"
    ),
    "summary": PromptTemplate(
        "Summarize the main points in a structured way.\n"
        "Context: {context_str}\n"
        "Question: {query_str}\n"
        "Summary:"
    )
}

# Frāzes, kas norāda, ka atbilde netika atrasta
NOT_FOUND_PHRASES = [
    "not provided",
    "not specified",
    "not mentioned",
    "no information",
    "cannot be found",
    "not available",
    "not possible"
]

def detect_query_type(question: str) -> str:
    q = question.lower()
    if any(v in q for v in ["compare", "difference", "contrast", "versus", "vs"]):
        return "comparison"
    elif any(v in q for v in ["summarize", "summary", "main points", "overview"]):
        return "summary"
    else:
        return "fact"

def answer_found(response_text: str) -> bool:
    # Pārbauda vai atbilde satur "nav atrasts" frāzes
    text = response_text.lower()
    return not any(phrase in text for phrase in NOT_FOUND_PHRASES)

async def smart_query(question: str) -> str:
    try:
        query_type = detect_query_type(question)

        # Salīdzināšanai lielāks top_k, lai atrastu abus dokumentus
        top_k = 6 if query_type == "comparison" else 3

        query_engine = indekss.as_query_engine(
            similarity_top_k=top_k,
            text_qa_template=PROMPTS[query_type]
        )

        result = query_engine.query(question)

        # Pārbauda vai atbilde tika atrasta
        if not answer_found(result.response):
            return (
                f"Query type: {query_type}\n"
                f"{'─' * 50}\n"
                f"Answer: No relevant information found in the documents.\n"
                f"{'─' * 50}\n"
                f"Suggestion: Try rephrasing your question."
            )

        sources = []
        for node in result.source_nodes:
            file_name = node.metadata.get("file_name", "unknown")
            if file_name not in sources:
                sources.append(file_name)

        # Strukturēta atbilde
        response = (
            f"Query type: {query_type}\n"
            f"{'─' * 50}\n"
            f"Answer: {result.response}\n"
            f"{'─' * 50}\n"
            f"Sources used:\n"
            + "\n".join(f"  • {s}" for s in sources)
        )
        return response

    except Exception as e:
        return f"Error: {str(e)}"

print("smart_query() funkcija gatava!")

smart_query() funkcija gatava!


In [12]:
# 5. Testa jautājumi
import asyncio

async def run_tests():
    # Faktu jautājums
    print(await smart_query("What is the projected deficit in the 2023 Canadian budget?"))
    print()

    # Salīdzināšanas jautājums
    print(await smart_query("Compare the main topics of the Canadian budget document and the AutoGen Studio document."))
    print()

    # Kopsavilkuma jautājums
    print(await smart_query("Give a summary of the main points in the paul graham essay."))

await run_tests()

Query type: fact
──────────────────────────────────────────────────
Answer: The projected deficit in the 2023 Canadian budget is $40.1 billion.
──────────────────────────────────────────────────
Sources used:
  • 2023_canadian_budget.pdf

Query type: comparison
──────────────────────────────────────────────────
Answer: Repeat.
──────────────────────────────────────────────────
Sources used:
  • 2023_canadian_budget.pdf
  • ag-studio.pdf

Query type: summary
──────────────────────────────────────────────────
Answer: The essay discusses Paul Graham's journey from working on a new Lisp dialect called Arc to starting an investment firm called Y Combinator. He reflects on the power of online publishing and the shift in the publishing landscape, leading him to write essays and eventually start Y Combinator. The essay also touches on his experiences with painting, building web applications, and the founding of Aspra. Ultimately, the essay highlights the importance of working on projects that 

In [13]:
# 6. top_k eksperiments
async def top_k_experiment():
    question = "What is the projected deficit in the 2023 Canadian budget?"
    results = {}

    for top_k in [1, 3, 5, 7]:
        query_engine = indekss.as_query_engine(
            similarity_top_k=top_k,
            text_qa_template=PROMPTS["fact"]
        )
        result = query_engine.query(question)
        sources = list(set(
            node.metadata.get("file_name", "unknown")
            for node in result.source_nodes
        ))
        results[top_k] = {
            "answer": result.response,
            "sources_count": len(result.source_nodes),
            "unique_docs": len(sources)
        }

    print("top_k eksperiments — jautājums par Kanādas budžeta deficītu")
    print("=" * 60)
    for top_k, data in results.items():
        print(f"top_k={top_k}")
        print(f"  Atrasti fragmenti : {data['sources_count']}")
        print(f"  Unikāli dokumenti : {data['unique_docs']}")
        print(f"  Atbilde           : {data['answer'][:80]}...")
        print()

await top_k_experiment()

top_k eksperiments — jautājums par Kanādas budžeta deficītu
top_k=1
  Atrasti fragmenti : 1
  Unikāli dokumenti : 1
  Atbilde           : The projected deficit in the 2023 Canadian budget is $40.1 billion....

top_k=3
  Atrasti fragmenti : 3
  Unikāli dokumenti : 1
  Atbilde           : The projected deficit in the 2023 Canadian budget is $40.1 billion....

top_k=5
  Atrasti fragmenti : 5
  Unikāli dokumenti : 2
  Atbilde           : The projected deficit in the 2023 Canadian budget is $40.1 billion....

top_k=7
  Atrasti fragmenti : 7
  Unikāli dokumenti : 2
  Atbilde           : The projected deficit in the 2023 Canadian budget is $40.1 billion....



# 7. Sistēmas apraksts

SISTĒMAS APRAKSTS
=================

Sistēma ir uzlabots RAG asistents, kas balstīts uz LlamaIndex ietvaru.
Tā analizē dokumentu kolekciju un sniedz strukturētas atbildes uz trīs jautājumu tipiem.

DATU PLŪSMA
-----------
1. SimpleDirectoryReader ielādē dokumentus (PDF un TXT formāti)
2. VectorStoreIndex pārveido dokumentus vektoros un saglabā indeksā
3. detect_query_type() nosaka jautājuma tipu pēc atslēgvārdiem
4. smart_query() izvēlas atbilstošu prompt un top_k parametru
5. query_engine meklē dokumentos un ģenerē atbildi
6. answer_found() pārbauda vai atbilde tika atrasta
7. Rezultāts tiek atgriezts strukturētā formātā ar avotiem

JAUTĀJUMU TIPI
--------------
- fact       : faktu jautājumi.            top_k=3
- comparison : salīdzināšanas jautājumi.   top_k=6
- summary    : kopsavilkuma jautājumi.     top_k=3

KĻŪDU APSTRĀDE
--------------
Ja atbilde satur frāzes "not provided", "not specified", "no information" u.c.
sistēma atpazīst, ka informācija nav atrasta un atgriež ieteikumu pārfrāzēt jautājumu.

TOP_K EKSPERIMENTA SECINĀJUMI
-----------------------------
top_k=1  →  1 fragments,  1 dokuments
top_k=3  →  3 fragmenti,  1 dokuments
top_k=5  →  5 fragmenti,  2 dokumenti
top_k=7  →  7 fragmenti,  2 dokumenti

Secinājums: Faktu jautājumiem pietiek ar top_k=3 — atbilde precīza jau ar 1 fragmentu.
Salīdzināšanas jautājumiem vajag top_k=6, lai atrastu fragmentus no vairākiem dokumentiem.

IZMANTOTIE DOKUMENTI
--------------------
- 2023_canadian_budget.pdf  : Kanādas federālais budžets 2023. gadam
- ag-studio.pdf             : AutoGen Studio zinātniskais raksts
- paul_graham_essay.txt     : Paul Graham eseja par programmēšanu un startapiem
